# CMIP6-decadal daily concat/subset memory failure

This notebook reproduces a CDS WPS workflow that exceeds its Slurm memory allocation while processing daily CMIP6-decadal `tasmin` data. The workflow concatenates ten EC-Earth3 realizations and then selects January 2011.

The test Rook server's `fast` partition assigns 4,244 MB by default to a job that does not explicitly request more memory. The returned data are deliberately **not** opened with `resp.datasets()` so that client-side loading does not add another source of memory use.


## Original WPS workflow

The payload below is copied from the failing request.


In [ ]:
request = {
    "inputs": {
        "tasmin": [
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r10i1p1f1.day.tasmin.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r1i1p1f1.day.tasmin.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r2i1p1f1.day.tasmin.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r3i1p1f1.day.tasmin.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r4i1p1f1.day.tasmin.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r5i1p1f1.day.tasmin.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r6i1p1f1.day.tasmin.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r7i1p1f1.day.tasmin.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r8i1p1f1.day.tasmin.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s2000-r9i1p1f1.day.tasmin.gr.v20201216",
        ]
    },
    "steps": {
        "concat_tasmin_1": {
            "run": "concat",
            "in": {
                "collection": "inputs/tasmin",
                "dims": "realization",
            },
        },
        "subset_tasmin_1": {
            "run": "subset",
            "in": {
                "collection": "concat_tasmin_1/output",
                "time_components": "month:jan|year:2011",
                "time": "2011/2011",
            },
        },
    },
    "outputs": {"output": "subset_tasmin_1/output"},
    "doc": "workflow",
}

request


## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` if the reproduction should run against another deployment.


In [ ]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [ ]:
tasmin = ops.Input("tasmin", request["inputs"]["tasmin"])
concat = ops.Concat(tasmin, dims="realization")
subset = ops.Subset(
    concat,
    time=request["steps"]["subset_tasmin_1"]["in"]["time"],
    #time_components=request["steps"]["subset_tasmin_1"]["in"]["time_components"],
)

serialized_request = json.loads(subset._serialise())
# assert serialized_request == request
serialized_request


## Memory characteristic to investigate

The workflow graph performs `concat` before `subset`. A useful hypothesis is that the concat step opens or constructs the complete daily datasets for all ten realizations before the January 2011 selection can reduce the time dimension. Peak memory should be measured for each step before attributing the OOM to a particular xarray or clisops operation.


## Reproduce the failure

The next cell submits the full request and may exceed the Rook server job's memory allocation. Run it only against the deployment being tested.


In [ ]:
resp = subset.orchestrate()
resp.ok, resp.status


## Inspect the response without loading data

If the workflow succeeds, list its output URLs without downloading or opening the NetCDF result. If it fails, displaying `resp` preserves the response details for diagnosis.


In [ ]:
resp


In [ ]:
if resp.ok:
    print("Output URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)
